# Klasifikasi Spesies Penguin: Training + Streamlit di Google Colab

**Target:** `species` (Adelie, Chinstrap, Gentoo) · **Data:** `kasus03_penguins.csv` (344 baris)

Urutan notebook: **(1)** setup & upload data → **(2)** tulis modul proyek → **(3)** audit data → **(4)** training & evaluasi → **(5)** cek model tersimpan → **(6)** jalankan aplikasi Streamlit.

Aturan yang dipatuhi: split train/test **sebelum** imputasi/scaling/encoding · pemilihan model dengan cross-validation di train · data test dievaluasi **satu kali** · model di `models/` · tanpa kredensial/path absolut.

## 1. Setup

In [ ]:
!pip install -q "streamlit>=1.50"
import os
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

Upload `kasus03_penguins.csv` (dilewati jika file sudah ada di `data/`).

In [ ]:
if not os.path.exists("data/kasus03_penguins.csv"):
    from google.colab import files
    up = files.upload()                      # pilih kasus03_penguins.csv
    name = next(iter(up))
    os.replace(name, "data/kasus03_penguins.csv")
print(os.listdir("data"))

## 2. Modul proyek
Tiga file ditulis ke direktori kerja: `features.py` (definisi kolom + fitur rasio), `train.py` (audit → split → CV → evaluasi → simpan), dan `app.py` (Streamlit). `features.py` harus ikut ada saat model dimuat karena pipeline memakai fungsi `add_features`.

In [ ]:
%%writefile features.py
import numpy as np
import pandas as pd

TARGET = "species"

# Kolom mentah yang dipakai sebagai fitur (input aplikasi = kolom-kolom ini).
RAW_NUM = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
RAW_CAT = ["island", "sex"]
INPUT_COLS = RAW_CAT + RAW_NUM

# Fitur turunan (dibuat di dalam pipeline, bukan oleh pengguna).
ENGINEERED = ["bill_ratio", "flipper_mass_ratio"]

# Kolom yang sengaja dibuang dari fitur beserta alasannya.
DROPPED = {
    "year": "Tahun pengamatan bukan ciri biologis penguin; hanya penanda waktu "
            "pencatatan. Distribusi spesies per tahun hampir sama sehingga tidak "
            "informatif, dan berisiko membuat model menghafal artefak sampling.",
}


def add_features(X: pd.DataFrame) -> pd.DataFrame:
    """Tambah fitur rasio. Aman untuk NaN dan pembagian nol."""
    X = X.copy()
    # Bentuk paruh: ramping-panjang (Gentoo/Chinstrap) vs pendek-tebal (Adelie).
    X["bill_ratio"] = X["bill_length_mm"] / X["bill_depth_mm"]
    # Panjang sirip relatif terhadap massa (massa dalam kg agar skalanya wajar).
    X["flipper_mass_ratio"] = X["flipper_length_mm"] / (X["body_mass_g"] / 1000.0)
    X[ENGINEERED] = X[ENGINEERED].replace([np.inf, -np.inf], np.nan)
    return X

In [ ]:
%%writefile train.py
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, log_loss, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, RepeatedStratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from features import DROPPED, ENGINEERED, INPUT_COLS, RAW_CAT, RAW_NUM, TARGET, add_features

# Semua path relatif terhadap lokasi file ini (tidak ada path absolut).
BASE_DIR = Path(__file__).resolve().parent
DATA_PATH = BASE_DIR / "data" / "kasus03_penguins.csv"
MODEL_DIR = BASE_DIR / "models"
MODEL_PATH = MODEL_DIR / "penguin_pipeline.joblib"

SEED = 42
TEST_SIZE = 0.2

# Batas kewajaran biologis (untuk audit nilai tidak wajar).
PLAUSIBLE = {
    "bill_length_mm": (25, 70),
    "bill_depth_mm": (8, 25),
    "flipper_length_mm": (150, 260),
    "body_mass_g": (2000, 7500),
}


# --------------------------------------------------------------------------- audit
def audit(df: pd.DataFrame) -> dict:
    """Audit deskriptif: duplikat, missing, nilai tidak wajar, tipe data."""
    report = {
        "n_rows": int(len(df)),
        "dtypes": {c: str(t) for c, t in df.dtypes.items()},
        "duplicate_rows": int(df.duplicated().sum()),
        "missing": {c: int(v) for c, v in df.isna().sum().items() if v > 0},
        "rows_all_measurements_missing": int(df[RAW_NUM].isna().all(axis=1).sum()),
        "implausible_values": {},
        "class_counts": df[TARGET].value_counts().to_dict(),
        "unexpected_categories": {},
    }
    for col, (lo, hi) in PLAUSIBLE.items():
        bad = df[col].notna() & ~df[col].between(lo, hi)
        report["implausible_values"][col] = int(bad.sum())
    for col in RAW_CAT:
        report["unexpected_categories"][col] = sorted(df[col].dropna().unique().tolist())
    return report


def load_data() -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH)
    # Pastikan kolom kategorikal bertipe object & nilai dinormalisasi (huruf kecil, tanpa spasi).
    for col in RAW_CAT:
        df[col] = df[col].astype("object").where(df[col].isna(), df[col].astype(str).str.strip().str.lower())
    df[TARGET] = df[TARGET].astype(str).str.strip()
    return df


# ------------------------------------------------------------------------ pipeline
def make_pipeline(model, engineered: bool = True, extra_num=(), use_island: bool = True) -> Pipeline:
    """Pipeline lengkap: feature engineering -> imputasi/scaling/encoding -> model.

    Semua statistik (median, mean/std, kategori, modus) dipelajari di .fit(),
    sehingga hanya dihitung dari data yang diberikan ke fit (= data train / fold train).
    """
    num_cols = RAW_NUM + (ENGINEERED if engineered else []) + list(extra_num)
    transformers = [
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), num_cols),
        # sex: NaN jadi kategori "unknown" (informasi 'tidak tercatat' dipertahankan,
        # tidak memalsukan jenis kelamin).
        ("sex", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="unknown")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), ["sex"]),
    ]
    if use_island:
        transformers.append(
            ("island", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                 ("oh", OneHotEncoder(handle_unknown="ignore"))]), ["island"]))
    return Pipeline([
        ("features", FunctionTransformer(add_features)),
        ("prep", ColumnTransformer(transformers, remainder="drop")),
        ("clf", model),
    ])


def candidate_models() -> dict:
    """Tiga algoritma + grid hiperparameter kecil (dicari lewat CV di data train)."""
    return {
        "Logistic Regression": (
            LogisticRegression(max_iter=5000, random_state=SEED),
            {"clf__C": [0.1, 1, 10]},
        ),
        "Random Forest": (
            RandomForestClassifier(random_state=SEED),
            {"clf__n_estimators": [200, 500], "clf__max_depth": [None, 5],
             "clf__min_samples_leaf": [1, 3]},
        ),
        "K-Nearest Neighbors": (
            KNeighborsClassifier(),
            {"clf__n_neighbors": [3, 5, 9, 15], "clf__weights": ["uniform", "distance"]},
        ),
    }


# ---------------------------------------------------------------------------- main
def main():
    MODEL_DIR.mkdir(exist_ok=True)
    df = load_data()

    # 1) AUDIT -------------------------------------------------------------------
    audit_report = audit(df)
    print("=== AUDIT DATA ===")
    print(json.dumps(audit_report, indent=2, default=str))

    # 2) Buang baris tanpa informasi sama sekali (semua pengukuran kosong).
    #    Ini aturan tetap per-baris (bukan statistik dari data), jadi aman dilakukan sebelum split.
    empty = df[RAW_NUM].isna().all(axis=1)
    print(f"\nBaris dibuang karena seluruh pengukuran kosong: {int(empty.sum())}")
    df = df.loc[~empty].reset_index(drop=True)

    # 3) SPLIT dulu, sebelum imputasi/scaling/encoding --------------------------
    X, y = df[INPUT_COLS], df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
    print(f"Train: {len(X_train)} | Test: {len(X_test)}")
    print("Missing di train (sebelum imputasi):", X_train.isna().sum()[lambda s: s > 0].to_dict())

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)

    # 4a) Ablasi fitur (CV di train saja) --------------------------------------
    print("\n=== ABLASI FITUR (CV f1_macro, data train) ===")
    ablation_defs = {
        "A. Mentah (4 numerik + island + sex)": dict(engineered=False),
        "B. A + fitur rasio  [DIPILIH]": dict(engineered=True),
        "C. B + year": dict(engineered=True, extra_num=("year",)),
        "D. B tanpa island": dict(engineered=True, use_island=False),
    }
    ablation_rows = []
    for name, kw in ablation_defs.items():
        row = {"feature_set": name}
        for mname, mdl in {"LogReg": LogisticRegression(max_iter=5000, random_state=SEED),
                           "RandForest": RandomForestClassifier(n_estimators=300, random_state=SEED)}.items():
            Xtr = X_train.copy()
            if "year" in kw.get("extra_num", ()):
                Xtr["year"] = df.loc[X_train.index, "year"]
            s = cross_val_score(make_pipeline(mdl, **kw), Xtr, y_train, cv=cv, scoring="f1_macro")
            row[mname] = round(float(s.mean()), 4)
        ablation_rows.append(row)
    ablation = pd.DataFrame(ablation_rows)
    print(ablation.to_string(index=False))

    # 4b) Pemilihan model: GridSearchCV di train ---------------------------------
    print("\n=== PEMILIHAN MODEL (CV f1_macro, data train) ===")
    searches, cv_rows = {}, []
    for name, (mdl, grid) in candidate_models().items():
        gs = GridSearchCV(make_pipeline(mdl), grid, cv=cv, scoring="f1_macro", n_jobs=-1)
        gs.fit(X_train, y_train)
        searches[name] = gs
        i = gs.best_index_
        cv_rows.append({
            "model": name,
            "cv_f1_macro_mean": round(float(gs.cv_results_["mean_test_score"][i]), 4),
            "cv_f1_macro_std": round(float(gs.cv_results_["std_test_score"][i]), 4),
            "best_params": {k.replace("clf__", ""): v for k, v in gs.best_params_.items()},
        })
    cv_table = pd.DataFrame(cv_rows).sort_values("cv_f1_macro_mean", ascending=False)
    print(cv_table.to_string(index=False))

    best_name = cv_table.iloc[0]["model"]
    best_pipe = searches[best_name].best_estimator_  # sudah di-refit pada seluruh data train
    print(f"\nModel terpilih (berdasarkan CV): {best_name}")

    # 5) EVALUASI TEST — satu kali, hanya model terpilih ------------------------
    classes = list(best_pipe.classes_)
    pred = best_pipe.predict(X_test)
    proba = best_pipe.predict_proba(X_test)
    rep = classification_report(y_test, pred, labels=classes, output_dict=True, zero_division=0)
    test_metrics = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
        "f1_macro": float(f1_score(y_test, pred, average="macro")),
        "f1_weighted": float(f1_score(y_test, pred, average="weighted")),
        "roc_auc_ovr_macro": float(roc_auc_score(y_test, proba, multi_class="ovr",
                                                 average="macro", labels=classes)),
        "log_loss": float(log_loss(y_test, proba, labels=classes)),
    }
    per_class = {c: {k: float(rep[c][k]) for k in ("precision", "recall", "f1-score", "support")}
                 for c in classes}
    cm = confusion_matrix(y_test, pred, labels=classes).tolist()

    print("\n=== EVALUASI DATA TEST (sekali) ===")
    print(json.dumps(test_metrics, indent=2))
    print(classification_report(y_test, pred, labels=classes, zero_division=0))
    print("Confusion matrix (baris=aktual, kolom=prediksi):", classes)
    print(np.array(cm))

    # 6) SIMPAN pipeline + metadata ---------------------------------------------
    metadata = {
        "classes": classes,
        "input_cols": INPUT_COLS,
        "numeric_inputs": {  # rentang dari data TRAIN saja
            c: {"min": float(X_train[c].min()), "max": float(X_train[c].max()),
                "median": float(X_train[c].median())} for c in RAW_NUM},
        "categorical_inputs": {
            "island": sorted(X_train["island"].dropna().unique().tolist()),
            "sex": sorted(X_train["sex"].dropna().unique().tolist()) + ["unknown"],
        },
        "engineered_features": ENGINEERED,
        "dropped_columns": DROPPED,
        "best_model": best_name,
        "best_params": cv_table.iloc[0]["best_params"],
        "cv_table": cv_table.to_dict(orient="records"),
        "ablation": ablation.to_dict(orient="records"),
        "test_metrics": test_metrics,
        "per_class": per_class,
        "confusion_matrix": cm,
        "audit": audit_report,
        "n_train": int(len(X_train)), "n_test": int(len(X_test)),
        "train_class_counts": y_train.value_counts().to_dict(),
        "seed": SEED, "test_size": TEST_SIZE,
        "sklearn_version": sklearn.__version__,
        "python_version": platform.python_version(),
        "trained_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    joblib.dump({"pipeline": best_pipe, "metadata": metadata}, MODEL_PATH, compress=3)
    (MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, default=str))
    print(f"\nTersimpan: {MODEL_PATH.relative_to(BASE_DIR)}  &  models/metadata.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile app.py
from pathlib import Path

import altair as alt
import joblib
import pandas as pd
import sklearn
import streamlit as st

from features import add_features  # noqa: F401  (diperlukan joblib untuk memuat pipeline)

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "models" / "penguin_pipeline.joblib"

LABELS = {
    "bill_length_mm": "Panjang paruh (mm)",
    "bill_depth_mm": "Kedalaman paruh (mm)",
    "flipper_length_mm": "Panjang sirip (mm)",
    "body_mass_g": "Massa tubuh (gram)",
}
SEX_LABELS = {"male": "Jantan", "female": "Betina", "unknown": "Tidak diketahui"}

st.set_page_config(page_title="Klasifikasi Spesies Penguin", page_icon="🐧", layout="wide")


@st.cache_resource
def load_bundle():
    if not MODEL_PATH.exists():
        return None
    return joblib.load(MODEL_PATH)


bundle = load_bundle()
st.title("🐧 Klasifikasi Spesies Penguin")

if bundle is None:
    st.error("Model belum ditemukan di `models/penguin_pipeline.joblib`. "
             "Jalankan `python train.py` terlebih dahulu.")
    st.stop()

pipe, meta = bundle["pipeline"], bundle["metadata"]
classes = meta["classes"]

if meta["sklearn_version"] != sklearn.__version__:
    st.warning(f"Model dilatih dengan scikit-learn {meta['sklearn_version']}, sedangkan environment "
               f"ini memakai {sklearn.__version__}. Jika muncul error/hasil aneh, jalankan ulang `train.py`.")

tab_pred, tab_eval, tab_info = st.tabs(["🔮 Prediksi", "📊 Evaluasi model", "🧾 Data & fitur"])

# ------------------------------------------------------------------ TAB PREDIKSI
with tab_pred:
    st.caption("Masukkan data mentah hasil pengamatan. Imputasi, scaling, encoding, dan fitur rasio "
               "diproses otomatis di dalam pipeline.")
    with st.form("input_form"):
        c1, c2 = st.columns(2)
        with c1:
            island = st.selectbox("Pulau pengamatan", meta["categorical_inputs"]["island"],
                                  format_func=str.title)
            sex = st.selectbox("Jenis kelamin", meta["categorical_inputs"]["sex"],
                               format_func=lambda v: SEX_LABELS.get(v, v),
                               index=meta["categorical_inputs"]["sex"].index("unknown"))
        values = {}
        with c2:
            for col, info in meta["numeric_inputs"].items():
                values[col] = st.number_input(LABELS[col], min_value=0.0, value=round(info["median"], 1),
                                              step=0.1 if col != "body_mass_g" else 10.0)
        submitted = st.form_submit_button("Prediksi spesies", type="primary")

    if submitted:
        row = pd.DataFrame([{"island": island, "sex": sex, **values}])[meta["input_cols"]]
        proba = pipe.predict_proba(row)[0]
        order = proba.argsort()[::-1]
        top, top_p = classes[order[0]], proba[order[0]]

        # peringatan jika input di luar rentang data latih
        out_of_range = [LABELS[c] for c, i in meta["numeric_inputs"].items()
                        if not (i["min"] <= values[c] <= i["max"])]
        if out_of_range:
            st.warning("Nilai di luar rentang data latih untuk: " + ", ".join(out_of_range)
                       + ". Prediksi kurang dapat diandalkan.")

        st.subheader(f"Prediksi: **{top}**")
        st.metric("Probabilitas tertinggi", f"{top_p:.1%}")
        if top_p < 0.60:
            st.info("Model kurang yakin (probabilitas tertinggi < 60%). Periksa kembali input.")

        prob_df = pd.DataFrame({"Spesies": classes, "Probabilitas": proba}).sort_values(
            "Probabilitas", ascending=False)
        left, right = st.columns(2)
        with left:
            st.dataframe(prob_df.style.format({"Probabilitas": "{:.2%}"}), hide_index=True,
                         width="stretch")
        with right:
            st.bar_chart(prob_df.set_index("Spesies"), y="Probabilitas")

        with st.expander("Fitur turunan yang dihitung otomatis"):
            derived = add_features(row)[meta["engineered_features"]]
            st.dataframe(derived.round(3), hide_index=True)

# --------------------------------------------------------------------- TAB EVALUASI
with tab_eval:
    tm = meta["test_metrics"]
    st.caption(f"Model terpilih: **{meta['best_model']}** · dilatih pada {meta['n_train']} data, "
               f"dievaluasi satu kali pada {meta['n_test']} data test (stratified split, seed {meta['seed']}).")
    m = st.columns(5)
    m[0].metric("Accuracy", f"{tm['accuracy']:.3f}")
    m[1].metric("Balanced accuracy", f"{tm['balanced_accuracy']:.3f}")
    m[2].metric("F1 macro", f"{tm['f1_macro']:.3f}")
    m[3].metric("ROC-AUC (OvR)", f"{tm['roc_auc_ovr_macro']:.3f}")
    m[4].metric("Log loss", f"{tm['log_loss']:.3f}")

    st.warning(f"Data test hanya {meta['n_test']} baris, sehingga skor sempurna di test tidak menjamin "
               "performa sempurna pada data baru. Gunakan hasil cross-validation (tabel di bawah) "
               "sebagai gambaran yang lebih stabil.")

    st.subheader("Per kelas (data test)")
    pc = pd.DataFrame(meta["per_class"]).T.rename(
        columns={"precision": "Precision", "recall": "Recall", "f1-score": "F1", "support": "Jumlah data"})
    pc["Jumlah data"] = pc["Jumlah data"].astype(int)
    st.dataframe(pc.style.format({"Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}"}),
                 width="stretch")

    st.subheader("Confusion matrix (data test)")
    cm = pd.DataFrame(meta["confusion_matrix"], index=classes, columns=classes)
    long = cm.reset_index(names="Aktual").melt("Aktual", var_name="Prediksi", value_name="Jumlah")
    base = alt.Chart(long).encode(x=alt.X("Prediksi:N", sort=classes), y=alt.Y("Aktual:N", sort=classes))
    heat = base.mark_rect().encode(color=alt.Color("Jumlah:Q", legend=None))
    txt = base.mark_text(fontSize=18).encode(
        text="Jumlah:Q",
        color=alt.condition(alt.datum.Jumlah > long["Jumlah"].max() / 2, alt.value("white"), alt.value("black")))
    st.altair_chart((heat + txt).properties(height=280), width="stretch")

    st.subheader("Perbandingan model (cross-validation di data train)")
    st.caption("5-fold × 3 pengulangan, skor F1 macro. Dipakai untuk memilih model; data test tidak dipakai.")
    cvt = pd.DataFrame(meta["cv_table"])
    cvt["best_params"] = cvt["best_params"].astype(str)
    st.dataframe(cvt.rename(columns={"model": "Model", "cv_f1_macro_mean": "F1 macro (rata-rata)",
                                     "cv_f1_macro_std": "Std", "best_params": "Hiperparameter terbaik"}),
                 hide_index=True, width="stretch")

    with st.expander("Apa arti metrik-metrik ini?"):
        st.markdown(
            "- **Accuracy**: proporsi seluruh prediksi yang benar. Bisa menyesatkan jika kelas tidak seimbang "
            "(Chinstrap hanya ±20% data).\n"
            "- **Balanced accuracy**: rata-rata recall tiap kelas, sehingga kelas kecil punya bobot setara.\n"
            "- **Precision**: dari semua yang diprediksi spesies X, berapa persen benar-benar X.\n"
            "- **Recall**: dari semua spesies X yang sebenarnya, berapa persen berhasil terdeteksi.\n"
            "- **F1 (macro)**: rata-rata harmonik precision & recall, dirata-ratakan sama rata antar kelas. "
            "Ini metrik utama untuk memilih model.\n"
            "- **ROC-AUC (OvR)**: kemampuan probabilitas memisahkan satu kelas dari sisanya; 1.0 = pemisahan sempurna.\n"
            "- **Log loss**: kualitas probabilitas; makin kecil makin baik, dan menghukum prediksi salah yang terlalu percaya diri.\n"
            "- **Confusion matrix**: baris = spesies sebenarnya, kolom = prediksi; angka di luar diagonal adalah kesalahan.")

# ------------------------------------------------------------------------ TAB INFO
with tab_info:
    a = meta["audit"]
    st.subheader("Audit data")
    st.markdown(
        f"- Jumlah baris awal: **{a['n_rows']}**; duplikat: **{a['duplicate_rows']}**\n"
        f"- Missing value: `{a['missing']}`\n"
        f"- Baris dengan seluruh pengukuran kosong: **{a['rows_all_measurements_missing']}** (dibuang, tidak ada "
        "informasi yang bisa dipakai atau diimputasi secara bermakna)\n"
        f"- Nilai tidak wajar (di luar batas biologis): `{a['implausible_values']}`\n"
        f"- Distribusi kelas: `{a['class_counts']}`")
    st.subheader("Keputusan fitur")
    st.markdown(
        "- **Fitur dipakai**: island, sex, bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g\n"
        "- **Fitur baru**: `bill_ratio` (panjang/kedalaman paruh) dan `flipper_mass_ratio` (sirip per kg massa)\n"
        "- **sex kosong** → kategori `unknown` (tidak memalsukan jenis kelamin); **numerik kosong** → median dari data train\n"
        "- **Scaling** (StandardScaler) untuk numerik; **One-hot** untuk island & sex")
    for col, reason in meta["dropped_columns"].items():
        st.markdown(f"- **Dibuang: `{col}`** — {reason}")
    st.subheader("Ablasi fitur (CV di data train, F1 macro)")
    st.dataframe(pd.DataFrame(meta["ablation"]).rename(columns={"feature_set": "Set fitur"}),
                 hide_index=True, width="stretch")
    st.caption("Catatan: `island` sangat informatif (mis. Torgersen hanya berisi Adelie di data ini), tetapi "
               "membuat model bergantung pada lokasi pengamatan; spesies di pulau baru bisa tidak terprediksi baik.")

In [ ]:
%%writefile requirements.txt
streamlit>=1.50
scikit-learn>=1.4
pandas>=2.0
numpy>=1.26
joblib>=1.3
altair>=5

## 3. Audit data
Bagian ini hanya **melihat** data (tidak mengubah apa pun). Hasil audit yang sama juga dicetak dan disimpan oleh `train.py`.

In [ ]:
import pandas as pd
df = pd.read_csv("data/kasus03_penguins.csv")
print(df.shape); display(df.head())
print("\nTipe data:"); print(df.dtypes)
print("\nDuplikat:", df.duplicated().sum())
print("\nMissing per kolom:"); print(df.isna().sum()[lambda s: s > 0])
print("\nBaris yang seluruh pengukurannya kosong:")
display(df[df[["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]].isna().all(axis=1)])
display(df.describe().T)

**Temuan & keputusan**

| Aspek | Temuan | Keputusan |
|---|---|---|
| Duplikat | 0 baris | Tidak ada yang dibuang |
| Tipe data | Numerik bertipe float, `sex`/`island` kategorikal, `year` integer | Kategorikal di-one-hot; numerik di-scale |
| Missing numerik | 2 baris, **keempat pengukuran kosong sekaligus** | Baris dibuang: tidak ada informasi untuk diimputasi secara bermakna. Ini aturan per-baris, bukan statistik, sehingga aman dilakukan sebelum split |
| Missing `sex` | 11 baris | Diisi kategori `unknown` di dalam pipeline (tidak menebak jenis kelamin, dan "tidak tercatat" dipertahankan sebagai informasi) |
| Nilai tidak wajar | 0 nilai di luar batas biologis | Tidak ada yang dibuang; variasi antarspesies adalah sinyal, bukan outlier |
| Kelas | Adelie 152, Gentoo 124, Chinstrap 68 | Split **stratified**; metrik utama F1 macro |

Setelah baris kosong dibuang, di data train **hanya `sex` yang masih punya missing**. Imputer median untuk kolom numerik tetap ada di pipeline sebagai pengaman untuk input baru (statistiknya dari data train saja).

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for sp, g in df.groupby("species"):
    ax[0].scatter(g.bill_length_mm, g.bill_depth_mm, label=sp, alpha=.7)
    ax[1].scatter(g.flipper_length_mm, g.body_mass_g, label=sp, alpha=.7)
    ax[2].scatter(g.bill_length_mm / g.bill_depth_mm, g.flipper_length_mm, label=sp, alpha=.7)
ax[0].set(xlabel="bill_length_mm", ylabel="bill_depth_mm", title="Paruh")
ax[1].set(xlabel="flipper_length_mm", ylabel="body_mass_g", title="Sirip vs massa")
ax[2].set(xlabel="bill_ratio (panjang/kedalaman)", ylabel="flipper_length_mm", title="Fitur rasio")
ax[0].legend(); plt.tight_layout(); plt.show()
print(pd.crosstab(df.island, df.species)); print(); print(pd.crosstab(df.year, df.species))

**Keputusan fitur**

- **`bill_ratio` = panjang / kedalaman paruh** (diminta). Adelie paruhnya pendek-tebal, Gentoo panjang-ramping, jadi rasio merangkum bentuk paruh dalam satu angka. Grafik ketiga menunjukkan pemisahan kelas yang lebih jelas.
- **`flipper_mass_ratio`** (tambahan): panjang sirip relatif terhadap massa (kg), ukuran proporsi tubuh yang tidak bergantung pada ukuran absolut.
- **`year` → dibuang.** Tahun hanyalah penanda waktu pencatatan, bukan ciri biologis, dan sebaran spesies per tahun hampir sama. Hasil ablasi di bawah juga menunjukkan tidak ada keuntungan berarti jika `year` dimasukkan (selisihnya jauh di bawah simpangan baku CV), jadi tidak ada alasan menanggung risiko model menghafal artefak sampling.
- **`island` → dipertahankan**, tetapi dengan catatan: sangat informatif (Torgersen hanya berisi Adelie, Chinstrap hanya di Dream, Gentoo hanya di Biscoe), namun membuat model bergantung pada lokasi. Pada pulau baru model tidak bisa mengandalkan fitur ini.
- Tidak ada kolom ID atau kolom "hasil setelah kejadian" di dataset ini, jadi selain `year` tidak ada yang dibuang.

## 4. Training & evaluasi
`train.py` menjalankan: split stratified 80/20 → ablasi fitur (CV di train) → `GridSearchCV` untuk Logistic Regression, Random Forest, dan KNN (5-fold × 3 pengulangan, skor F1 macro) → evaluasi **satu kali** model terpilih di data test → simpan ke `models/`.

Pipeline: `add_features` → `ColumnTransformer` (numerik: median imputer + StandardScaler; `sex`: imputer `unknown` + one-hot; `island`: one-hot) → model. Karena semuanya berada dalam satu pipeline, setiap statistik dipelajari hanya dari data train (atau dari fold train saat CV).

In [ ]:
!python train.py

### Cara membaca metrik

- **Accuracy**: proporsi prediksi benar; bisa menipu bila kelas tidak seimbang.
- **Balanced accuracy**: rata-rata recall per kelas, sehingga kelas kecil (Chinstrap) sama pentingnya.
- **Precision / Recall / F1 per kelas**: ketepatan prediksi vs. kelengkapan deteksi tiap spesies.
- **F1 macro**: rata-rata F1 antar kelas tanpa bobot jumlah data, **metrik utama pemilihan model** karena ada ketimpangan kelas.
- **ROC-AUC (one-vs-rest)** dan **log loss**: menilai kualitas probabilitas, bukan hanya label.
- **Confusion matrix**: memperlihatkan spesies mana yang tertukar dengan spesies mana.

**Catatan penting:** data test hanya 69 baris, dan skor test yang sempurna belum tentu berarti model sempurna di dunia nyata. Selisih ketiga model di CV (≈0,99–0,997) lebih kecil daripada simpangan baku antar-fold, sehingga pemilihan Random Forest sebaiknya dibaca sebagai "ketiganya setara, Random Forest unggul tipis", bukan kemenangan telak.

## 5. Verifikasi model tersimpan

In [ ]:
import joblib, pandas as pd
from features import add_features        # diperlukan untuk memuat pipeline
bundle = joblib.load("models/penguin_pipeline.joblib")
pipe, meta = bundle["pipeline"], bundle["metadata"]
print("Model:", meta["best_model"], "| classes:", meta["classes"])
print("Metrik test:", {k: round(v, 3) for k, v in meta["test_metrics"].items()})

contoh = pd.DataFrame([{"island": "biscoe", "sex": "female", "bill_length_mm": 47.5,
                        "bill_depth_mm": 15.0, "flipper_length_mm": 217.0, "body_mass_g": 5000.0}])
print(dict(zip(pipe.classes_, pipe.predict_proba(contoh)[0].round(3))))

## 6. Jalankan aplikasi Streamlit di Colab
Colab tidak punya URL publik, jadi aplikasi dibuka lewat tunnel **localtunnel**. Jalankan sel di bawah, lalu:
1. Klik URL `https://....loca.lt` yang tercetak.
2. Jika diminta *Tunnel Password*, isi dengan alamat IP yang tercetak di atasnya.

Sel tunnel berjalan terus (blocking). Hentikan dengan tombol stop pada sel.

In [ ]:
import subprocess, time, urllib.request

# Jalankan Streamlit di latar belakang (log ke streamlit.log)
log = open("streamlit.log", "w")
proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=log, stderr=log)
time.sleep(5)
print("Streamlit berjalan:", proc.poll() is None)

# Password tunnel = IP publik runtime Colab
print("Tunnel password (IP):", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())

In [ ]:
!npx -y localtunnel --port 8501